# Tutorial 7: Train NicheTrans on 10x Xenium data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_breast_cancer import Breast_cancer

from utils.utils import *
from utils.utils_training_breast_cancer import train, test
from utils.utils_dataloader import *
from prior_AddOn.gene_embedding_loader import load_static_gene_prior
from prior_AddOn.gene_prior_filter import filter_dataset_by_gene_prior

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_breast_cancer.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.2, dropout_rate=0.2, workers=4, adata_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/HBC_rep1_cell_nucleus_3channel_strength_mean.h5ad', coordinate_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/cells.csv.gz', ct_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/Cell_Barcode_Type_Matrices.xlsx', max_epoch=40, stepsize=20, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataset
dataset = Breast_cancer(adata_path=args.adata_path, coordinate_path=args.coordinate_path, ct_path=args.ct_path)

# load and apply static gene priors before creating dataloaders/model
prior_model = "scgpt"
priors = load_static_gene_prior(
    source_panel=dataset.source_panel,
    models=(prior_model,),
    species="human",
    root="prior_AddOn/gene_embeddings",
)
dataset, priors, filter_info = filter_dataset_by_gene_prior(
    dataset=dataset,
    priors=priors,
    prior_model=prior_model,
)
print(
    f"Gene prior filtering ({prior_model}): kept "
    f"{len(filter_info['filtered_source_panel'])}/"
    f"{len(filter_info['original_source_panel'])} genes; "
    f"removed {len(filter_info['removed_genes'])}."
)

# create the dataloaders after filtering so feature dimensions stay aligned
trainloader, testloader = breast_cancer_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.protein_length
model = NicheTrans(
    source_length=source_dimension,
    target_length=target_dimension,
    noise_rate=args.noise_rate,
    dropout_rate=args.dropout_rate,
    priors=priors,
    prior_model=prior_model,
)
model = nn.DataParallel(model).cuda()


------Calculating spatial graph...
The graph contains 1185564 edges, 98797 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 827796 edges, 68983 cells.
12.0000 neighbors per cell on average.
=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  98797 spots, 98659 positive CD20, 84043 positive HER2 
  test     |  68983 spots, 67600 positive CD20, 36904 positive HER2 
  ------------------------------
Gene prior filtering (scgpt): kept 311/313 genes; removed 2.


### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.MSELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader)
    if args.stepsize > 0: scheduler.step()
    ################
    
pearson = test(model, testloader)
torch.save(model.state_dict(), 'NicheTrans_breast_cancer_last.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/40


Batch 3087/3087	 Loss 0.285688 (0.311009)
==> Epoch 2/40
Batch 3087/3087	 Loss 0.200217 (0.221161)
==> Epoch 3/40
Batch 3087/3087	 Loss 0.172656 (0.208824)
==> Epoch 4/40
Batch 3087/3087	 Loss 0.086396 (0.197454)
==> Epoch 5/40
Batch 3087/3087	 Loss 0.159808 (0.190684)
==> Epoch 6/40
Batch 3087/3087	 Loss 0.416575 (0.184472)
==> Epoch 7/40
Batch 3087/3087	 Loss 0.165161 (0.175511)
==> Epoch 8/40
Batch 3087/3087	 Loss 0.122637 (0.173468)
==> Epoch 9/40
Batch 3087/3087	 Loss 0.204211 (0.170766)
==> Epoch 10/40
Batch 3087/3087	 Loss 0.143111 (0.166835)
==> Epoch 11/40
Batch 3087/3087	 Loss 0.155931 (0.164901)
==> Epoch 12/40
Batch 3087/3087	 Loss 0.192023 (0.163244)
==> Epoch 13/40
Batch 3087/3087	 Loss 0.110895 (0.159648)
==> Epoch 14/40
Batch 3087/3087	 Loss 0.212371 (0.159060)
==> Epoch 15/40
Batch 3087/3087	 Loss 0.219619 (0.156982)
==> Epoch 16/40
Batch 3087/3087	 Loss 0.133020 (0.156514)
==> Epoch 17/40
Batch 3087/3087	 Loss 0.182487 (0.155454)
==> Epoch 18/40
Batch 3087/3087	 Loss 